In [9]:
import torch
import numpy as np
from skimage import io, segmentation, color
from skimage.graph import rag_mean_color
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse
from skimage.measure import regionprops
from skimage import img_as_float

def image_to_graph_with_superpixels(image_path, num_superpixels=200, compactness=10):
    # Load image and convert to RGB if it's grayscale
    image = io.imread(image_path)
    if len(image.shape) == 2:  # Grayscale image, convert to RGB
        image = np.stack([image] * 3, axis=-1)
    
    # Convert to float image and apply superpixel segmentation (SLIC)
    image_float = img_as_float(image)
    segments = segmentation.slic(image_float, n_segments=num_superpixels, compactness=compactness, start_label=1)
    
    # Create a graph where each superpixel is a node
    g = rag_mean_color(image, segments)
    
    # Node features: Use mean color of the superpixel as node feature
    node_features = []
    for region in regionprops(segments):
        # Extract mean color of each superpixel
        mean_color = np.mean(image[segments == region.label], axis=0)
        node_features.append(mean_color)
    
    node_features = np.array(node_features)
    
    # Define edges based on adjacency of superpixels
    edge_index = []
    for u, v, data in g.edges(data=True):
        edge_index.append([u-1, v-1])  # Adjust indices for zero-based indexing
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    # Convert node features into tensor format
    node_features = torch.tensor(node_features, dtype=torch.float)

    # Create the PyTorch Geometric data object
    data = Data(x=node_features, edge_index=edge_index)
    
    return data

# Example usage
image_path = 'C:\\Users\\SHAYANTAN BISWAS\\Pictures\\Academic Block'
graph_data = image_to_graph_with_superpixels(image_path)

print(graph_data)


FileNotFoundError: No such file: 'C:\Users\SHAYANTAN BISWAS\Pictures\Academic Block'